In [1]:
import pandas as pd
import numpy as np
import requests

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

from tqdm.auto import tqdm


# API endpoint
API_URL = "http://127.0.0.1:8000/api/v1/incident/analyze"

# Test dataset path
TEST_FILE = "final.csv"

# Read test dataset
test_df = pd.read_csv(TEST_FILE)

print("Test dataset shape:", test_df.shape)
print(test_df.head())

Test dataset shape: (1412, 33)
     host_received_timestamp_utc  device_time_ms  stage_elapsed_ms condition  \
0  2026-10-28T04:14:20.185+00:00           81178              6400    normal   
1  2026-10-28T04:14:20.385+00:00           81378              6600    normal   
2  2026-10-28T04:14:20.585+00:00           81578              6800    normal   
3  2026-10-28T04:14:20.785+00:00           81778              7000    normal   
4  2026-10-28T04:14:20.985+00:00           81978              7200    normal   

   motor_running motor_direction  speed_percent  pwm_duty_8bit  ina219_ok  \
0              1         forward             20             51          1   
1              1         forward             20             51          1   
2              1         forward             20             51          1   
3              1         forward             20             51          1   
4              1         forward             20             51          1   

   power_data_valid  ... 

In [2]:
SENSOR_COLUMNS = [
    "device_time_ms",
    "stage_elapsed_ms",
    "motor_running",
    "motor_direction",
    "speed_percent",

    "power_data_valid",
    "bus_voltage_v",
    "shunt_voltage_mv",
    "current_ma",
    "power_mw",

    "ds18b20_ok",
    "temperature_valid",
    "temperature_c",

    "mpu6050_ok",
    "mpu_data_valid",
    "accel_x_m_s2",
    "accel_y_m_s2",
    "accel_z_m_s2",
    "gyro_x_rad_s",
    "gyro_y_rad_s",
    "gyro_z_rad_s",

    "encoder_count",
    "encoder_delta",
    "encoder_counts_per_second",
    "encoder_counts_per_revolution",
     
    "overcurrent_fault",
    "last_motor_stop_reason"
]

In [3]:
test_df["condition"] = (
    test_df["condition"]
    .astype(str)
    .str.lower()
    .str.strip()
)

condition_mapping = {
    "normal": 0,
    "overheated": 1,
    "vibration_issue": 1
}

test_df["actual"] = test_df["condition"].map(condition_mapping)

# Check for unknown conditions
if test_df["actual"].isna().any():
    unknown = test_df.loc[
        test_df["actual"].isna(), "condition"
    ].unique()

    raise ValueError(f"Unknown conditions found: {unknown}")

test_df["actual"] = test_df["actual"].astype(int)

print(test_df["actual"].value_counts())

actual
0    989
1    423
Name: count, dtype: int64


In [4]:
results = []

for index, row in tqdm(test_df.iterrows(), total=len(test_df)):

    # Prepare sensor data only
    sensor_data = {}

    for col in SENSOR_COLUMNS:

        value = row[col]

        # Convert missing values to None
        if pd.isna(value):
            value = None

        # Convert NumPy data types to Python types
        elif isinstance(value, np.generic):
            value = value.item()

        sensor_data[col] = value

    # Get timestamp
    timestamp = None

    if "host_received_timestamp_utc" in test_df.columns:
        if pd.notna(row["host_received_timestamp_utc"]):
            timestamp = pd.to_datetime(
                row["host_received_timestamp_utc"],
                utc=True
            ).isoformat()

    # Build API request
    payload = {
        "incident_id": f"TEST-{index:05d}",
        "asset_id": "esp32-conveyor-01",
        "timestamp": timestamp,
        "sensor_data": sensor_data
    }

    try:
        response = requests.post(
            API_URL,
            json=payload,
            timeout=15
        )

        response.raise_for_status()

        prediction = response.json()

        results.append({
    "row_index": index,
    "condition": row["condition"],
    "actual": int(row["actual"]),
    "predicted": int(prediction["is_anomaly"]),

    # Original sensor readings
    "speed_percent": row["speed_percent"],
    "temperature_c": row["temperature_c"],
    "current_ma": row["current_ma"],
    "power_mw": row["power_mw"],
    "bus_voltage_v": row["bus_voltage_v"],
    "accel_x_m_s2": row["accel_x_m_s2"],
    "accel_y_m_s2": row["accel_y_m_s2"],
    "accel_z_m_s2": row["accel_z_m_s2"],
    "stage_elapsed_ms": row["stage_elapsed_ms"],

    # Model outputs
    "raw_score": prediction["raw_score"],
    "threshold": prediction["threshold"],
    "anomaly_score": prediction["anomaly_score"],

    "status": "success",
    "error": None
})
    except requests.exceptions.RequestException as exc:

        error_detail = str(exc)

        try:
            error_detail = response.json().get("detail", error_detail)
        except Exception:
            pass

        results.append({
            "row_index": index,
            "condition": row["condition"],
            "actual": int(row["actual"]),
            "predicted": None,
            "raw_score": None,
            "threshold": None,
            "anomaly_score": None,
            "is_anomaly": None,
            "status": "failed",
            "error": error_detail
        })


results_df = pd.DataFrame(results)

print(results_df.head())
print("\nAPI Results:")
print(results_df["status"].value_counts())

  0%|          | 0/1412 [00:00<?, ?it/s]

   row_index condition  actual  predicted  speed_percent  temperature_c  \
0          0    normal       0          0             20           26.5   
1          1    normal       0          0             20           26.5   
2          2    normal       0          0             20           26.5   
3          3    normal       0          0             20           26.5   
4          4    normal       0          0             20           26.5   

   current_ma  power_mw  bus_voltage_v  accel_x_m_s2  accel_y_m_s2  \
0       165.1    1098.0          7.148       -9.4128       -0.0970   
1       152.5    1088.0          7.116       -9.4200       -0.0898   
2       154.3    1022.0          7.148       -9.4056       -0.0802   
3       157.3    1120.0          7.128       -9.4164       -0.0634   
4       152.8    1036.0          7.140       -9.3937       -0.0874   

   accel_z_m_s2  stage_elapsed_ms  raw_score  threshold  anomaly_score  \
0       -0.2622              6400   0.997441    1.8202

In [5]:
# Keep successful API responses only
evaluation_df = results_df[
    results_df["status"] == "success"
].copy()

y_true = evaluation_df["actual"]
y_pred = evaluation_df["predicted"]

# Calculate evaluation metrics
accuracy = accuracy_score(y_true, y_pred)

precision = precision_score(
    y_true, y_pred, zero_division=0
)
recall = recall_score(
    y_true, y_pred, zero_division=0
)
f1 = f1_score(
    y_true, y_pred, zero_division=0
)
metrics_df = pd.DataFrame([{
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "F1-Score": f1
}])

metrics_df.round(4)

,Accuracy,Precision,Recall,F1-Score
0,0.9717,0.9848,0.9196,0.9511


In [6]:
cm = confusion_matrix(
    y_true,
    y_pred,
    labels=[0, 1]
)

cm_df = pd.DataFrame(
    cm,
    index=["Actual Normal", "Actual Anomaly"],
    columns=["Predicted Normal", "Predicted Anomaly"]
)

cm_df

,Predicted Normal,Predicted Anomaly
Actual Normal,983,6
Actual Anomaly,34,389


In [7]:
missed = results_df[(results_df["actual"] == 1) & (results_df["predicted"] == 0)].copy()
t = test_df.loc[missed["row_index"]]

missed["type"] = np.where(
    ((t["condition"] == "overheated") & ~(t["temperature_c"] >= 30)).values,
    "cold overheated (ambiguous)",
    np.where(((t["condition"] == "vibration_issue") & (t["motor_running"] == 0)).values,
             "idle vibration (ambiguous)", "REAL miss"))
print(missed["type"].value_counts())

type
cold overheated (ambiguous)    34
Name: count, dtype: int64


In [8]:
real = missed["type"].eq("REAL miss").values
print(t.loc[real, ["condition", "speed_percent", "temperature_c", "current_ma"]])

Empty DataFrame
Columns: [condition, speed_percent, temperature_c, current_ma]
Index: []


In [9]:
print(
    classification_report(
        y_true,
        y_pred,
        labels=[0, 1],
        target_names=["Normal", "Anomaly"],
        zero_division=0
    )
)

              precision    recall  f1-score   support

      Normal       0.97      0.99      0.98       989
     Anomaly       0.98      0.92      0.95       423

    accuracy                           0.97      1412
   macro avg       0.98      0.96      0.97      1412
weighted avg       0.97      0.97      0.97      1412



In [10]:
HOT_TEMP_C = 30.0
t = test_df.loc[results_df["row_index"]]
results_df["ambiguous"] = (
    ((t["condition"] == "overheated") & ~(t["temperature_c"] >= HOT_TEMP_C))
    | ((t["condition"] == "vibration_issue") & (t["motor_running"] == 0))
).values

scored = results_df[~results_df["ambiguous"]]
print(classification_report(scored["actual"], scored["predicted"],
                            target_names=["Normal", "Anomaly"]))
print("flagged among ambiguous rows:",
      results_df.loc[results_df["ambiguous"], "predicted"].mean())

              precision    recall  f1-score   support

      Normal       1.00      0.99      1.00       989
     Anomaly       0.98      1.00      0.99       311

    accuracy                           1.00      1300
   macro avg       0.99      1.00      0.99      1300
weighted avg       1.00      1.00      1.00      1300

flagged among ambiguous rows: 0.6964285714285714


In [11]:
ambiguous_df = results_df[results_df["ambiguous"]].copy()

print("Ambiguous rows:", len(ambiguous_df))

print("\nActual condition distribution:")
print(
    test_df.loc[ambiguous_df["row_index"], "condition"]
    .value_counts()
)

print("\nPredictions among ambiguous rows:")
print(
    ambiguous_df["predicted"].value_counts()
)

print("\nFlagged rate by condition:")
ambiguous_df["condition"] = (
    test_df.loc[ambiguous_df["row_index"], "condition"].values
)

print(
    ambiguous_df.groupby("condition")["predicted"].agg(
        ["count", "mean"]
    )
)

Ambiguous rows: 112

Actual condition distribution:
condition
overheated    112
Name: count, dtype: int64

Predictions among ambiguous rows:
predicted
1    78
0    34
Name: count, dtype: int64

Flagged rate by condition:
            count      mean
condition                  
overheated    112  0.696429


In [12]:
ambiguous_df = results_df[results_df["ambiguous"]].copy()

ambiguous_df["temperature_c"] = test_df.loc[
    ambiguous_df["row_index"], "temperature_c"
].values

print(
    ambiguous_df.groupby("predicted")["temperature_c"].agg(
        ["count", "min", "max", "mean", "median", "std"]
    )
)

           count    min    max       mean  median       std
predicted                                                  
0             34  24.75  28.00  27.529412   27.75  0.815951
1             78  24.75  29.75  27.839744   27.75  0.790543


In [13]:
ambiguous_df = results_df[results_df["ambiguous"]].copy()

sensor_features = [
    "speed_percent",
    "bus_voltage_v",
    "current_ma",
    "power_mw",
    "accel_x_m_s2",
    "accel_y_m_s2",
    "accel_z_m_s2",
    "gyro_x_rad_s",
    "gyro_y_rad_s",
    "gyro_z_rad_s",
    "encoder_counts_per_second",
]

sensor_data = test_df.loc[
    ambiguous_df["row_index"],
    sensor_features
].copy()

sensor_data["prediction"] = ambiguous_df["predicted"].values

comparison = sensor_data.groupby("prediction").mean().T

comparison.columns = ["Predicted Normal", "Predicted Anomaly"]

comparison["Difference"] = (
    comparison["Predicted Anomaly"] -
    comparison["Predicted Normal"]
)

print(comparison.round(3))

                           Predicted Normal  Predicted Anomaly  Difference
speed_percent                        47.353             45.385      -1.968
bus_voltage_v                         6.804              6.800      -0.003
current_ma                          201.712            186.344     -15.368
power_mw                           1363.412           1274.256     -89.155
accel_x_m_s2                         -9.430             -9.425       0.006
accel_y_m_s2                         -0.010             -0.007       0.003
accel_z_m_s2                          0.249              0.209      -0.040
gyro_x_rad_s                         -0.015             -0.016      -0.000
gyro_y_rad_s                         -0.028             -0.030      -0.001
gyro_z_rad_s                         -0.005             -0.005      -0.001
encoder_counts_per_second          1807.353           1722.115     -85.238


In [14]:
test_row = test_df.iloc[0]

sensor_data = {}

for col in SENSOR_COLUMNS:
    value = test_row[col]

    if pd.isna(value):
        value = None
    elif isinstance(value, np.generic):
        value = value.item()

    sensor_data[col] = value

payload = {
    "incident_id": "DEBUG-001",
    "asset_id": "esp32-conveyor-01",
    "timestamp": None,
    "sensor_data": sensor_data
}

response = requests.post(
    API_URL,
    json=payload,
    timeout=60
)

print("Status Code:", response.status_code)
print("Response:", response.text)

Status Code: 200
Response: {"incident_id":"DEBUG-001","asset_id":"esp32-conveyor-01","timestamp":"2026-11-24T09:29:09.887341Z","window":{"start":"2026-11-24T09:29:09.887341Z","end":"2026-11-24T09:29:09.887341Z","n_samples":1},"is_anomaly":false,"anomaly_score":0.353987147238636,"raw_score":0.9974405019283272,"threshold":1.8202903386095501,"model":{"name":"lof","version":"0.1","feature_version":"f1"},"top_features":[{"name":"gyro_z_rad_s","contribution":0.021647427981974388,"value":-0.0067,"baseline":-0.0053,"direction":"low","score_after_neutralizing":0.9757930739463528},{"name":"acceleration_magnitude","contribution":0.010349303698299228,"value":9.416950763384081,"baseline":9.435482870526553,"direction":"low","score_after_neutralizing":0.987091198230028},{"name":"accel_x_m_s2","contribution":0.010054203185538957,"value":-9.4128,"baseline":-9.4236,"direction":"high","score_after_neutralizing":0.9873862987427883},{"name":"gyro_x_rad_s","contribution":0.008914249511258632,"value":-0.016,

In [15]:
from sklearn.metrics import classification_report, confusion_matrix
import pandas as pd

# Add predictions to the final evaluation dataset
evaluation_df = test_df.copy()

evaluation_df["predicted"] = y_pred

# Convert predictions to readable labels if needed
evaluation_df["predicted_label"] = evaluation_df["predicted"].map({
    0: "normal",
    1: "anomaly"
})

evaluation_df["actual_label"] = (
    evaluation_df["condition"]
    .map({
        "normal": "normal",
        "overheated": "anomaly",
        "vibration_issue": "anomaly"
    })
)

# Evaluate each hidden speed separately
for speed in sorted(evaluation_df["speed_percent"].unique()):

    speed_df = evaluation_df[
        evaluation_df["speed_percent"] == speed
    ]

    print(f"\n{'='*50}")
    print(f"Speed: {speed}%")
    print(f"{'='*50}")

    print("\nConfusion Matrix:")

    print(confusion_matrix(
        speed_df["actual_label"],
        speed_df["predicted_label"],
        labels=["normal", "anomaly"]
    ))

    print("\nClassification Report:")

    print(classification_report(
        speed_df["actual_label"],
        speed_df["predicted_label"],
        labels=["normal", "anomaly"],
        zero_division=0
    ))

    print("\nPerformance by actual condition:")

    print(pd.crosstab(
        speed_df["condition"],
        speed_df["predicted_label"]
    ))


Speed: 18%

Confusion Matrix:
[[45  1]
 [ 0  0]]

Classification Report:
              precision    recall  f1-score   support

      normal       1.00      0.98      0.99        46
     anomaly       0.00      0.00      0.00         0

    accuracy                           0.98        46
   macro avg       0.50      0.49      0.49        46
weighted avg       1.00      0.98      0.99        46


Performance by actual condition:
predicted_label  anomaly  normal
condition                       
normal                 1      45

Speed: 19%

Confusion Matrix:
[[28  1]
 [ 0  0]]

Classification Report:
              precision    recall  f1-score   support

      normal       1.00      0.97      0.98        29
     anomaly       0.00      0.00      0.00         0

    accuracy                           0.97        29
   macro avg       0.50      0.48      0.49        29
weighted avg       1.00      0.97      0.98        29


Performance by actual condition:
predicted_label  anomaly  norma

In [16]:
# Filter missed overheating cases at 50% speed
missed_overheating = results_df[
    (results_df["speed_percent"] == 50) &
    (results_df["condition"] == "overheated") &
    (results_df["predicted"] == 0)
]

# Inspect their sensor readings and anomaly scores
columns_to_inspect = [
    "temperature_c",
    "current_ma",
    "power_mw",
    "bus_voltage_v",
    "accel_x_m_s2",
    "accel_y_m_s2",
    "accel_z_m_s2",
    "anomaly_score",
    "raw_score",
    "threshold"
]

print(missed_overheating[columns_to_inspect].describe())

       temperature_c  current_ma     power_mw  bus_voltage_v  accel_x_m_s2  \
count       31.00000   31.000000    31.000000      31.000000     31.000000   
mean        27.75000  206.422581  1394.387097       6.796000     -9.432190   
std          0.33541   31.091496   196.082581       0.023732      0.031439   
min         26.25000  147.800000  1000.000000       6.736000     -9.494200   
25%         27.75000  180.400000  1271.000000       6.780000     -9.450500   
50%         27.75000  217.700000  1426.000000       6.800000     -9.437900   
75%         28.00000  233.450000  1550.000000       6.816000     -9.418800   
max         28.00000  249.400000  1688.000000       6.824000     -9.369700   

       accel_y_m_s2  accel_z_m_s2  anomaly_score  raw_score     threshold  
count     31.000000     31.000000      31.000000  31.000000  3.100000e+01  
mean      -0.009084      0.228955       0.450039   1.499337  1.820290e+00  
std        0.063877      0.089872       0.030556   0.182149  9.028600

In [17]:
# Compare the three groups at 50% speed

groups = {
    "Normal": results_df[
        (results_df["speed_percent"] == 50) &
        (results_df["condition"] == "normal")
    ],

    "Detected Overheated": results_df[
        (results_df["speed_percent"] == 50) &
        (results_df["condition"] == "overheated") &
        (results_df["predicted"] == 1)
    ],

    "Missed Overheated": results_df[
        (results_df["speed_percent"] == 50) &
        (results_df["condition"] == "overheated") &
        (results_df["predicted"] == 0)
    ]
}

features = [
    "temperature_c",
    "current_ma",
    "power_mw",
    "bus_voltage_v",
    "accel_x_m_s2",
    "accel_y_m_s2",
    "accel_z_m_s2",
    "raw_score"
]

for group_name, group_data in groups.items():

    print("\n" + "=" * 50)
    print(group_name)
    print("Number of samples:", len(group_data))
    print("=" * 50)

    display(group_data[features].describe().loc[["mean", "std", "min", "max"]])


Normal
Number of samples: 204


,temperature_c,current_ma,power_mw,bus_voltage_v,accel_x_m_s2,accel_y_m_s2,accel_z_m_s2,raw_score
mean,25.617647,184.112255,1477.088235,8.028882,-9.426210,-0.026333,0.300197,1.210357
std,1.105687,31.870406,333.342487,1.898024,0.054358,0.046641,0.244525,0.183323
min,24.500000,113.300000,1010.000000,6.504000,-9.643800,-0.152000,-0.271700,0.967732
max,27.250000,261.800000,2680.000000,11.204000,-9.258400,0.099400,0.894200,1.722453



Detected Overheated
Number of samples: 133


,temperature_c,current_ma,power_mw,bus_voltage_v,accel_x_m_s2,accel_y_m_s2,accel_z_m_s2,raw_score
mean,33.584586,194.393233,1333.864662,6.825805,-9.424335,-0.013357,0.275280,5.970491
std,9.147409,30.335716,207.090259,0.073847,0.021101,0.041356,0.114447,6.844869
min,24.750000,148.000000,1002.000000,6.740000,-9.466700,-0.105300,0.079000,1.860954
max,73.000000,269.500000,1832.000000,7.116000,-9.369700,0.074200,0.615300,39.881888



Missed Overheated
Number of samples: 31


,temperature_c,current_ma,power_mw,bus_voltage_v,accel_x_m_s2,accel_y_m_s2,accel_z_m_s2,raw_score
mean,27.75000,206.422581,1394.387097,6.796000,-9.432190,-0.009084,0.228955,1.499337
std,0.33541,31.091496,196.082581,0.023732,0.031439,0.063877,0.089872,0.182149
min,26.25000,147.800000,1000.000000,6.736000,-9.494200,-0.147200,0.063400,1.111795
max,28.00000,249.400000,1688.000000,6.824000,-9.369700,0.112500,0.414200,1.802109


In [18]:
# Compare temperature distributions at 50% speed

temperature_analysis = (
    results_df[
        results_df["speed_percent"] == 50
    ]
    .groupby(["condition", "predicted"])["temperature_c"]
    .agg(["count", "mean", "std", "min", "max"])
)

display(temperature_analysis)

count       mean       std    min    max
condition       predicted                                          
normal          0            204  25.617647  1.105687  24.50  27.25
overheated      0             31  27.750000  0.335410  26.25  28.00
                1            133  33.584586  9.147409  24.75  73.00
vibration_issue 1             22  26.409091  0.675002  25.75  28.25

In [19]:
# Analyze elapsed time for overheating cases at 50% speed

overheated_50 = results_df[
    (results_df["speed_percent"] == 50) &
    (results_df["condition"] == "overheated")
].copy()

# Retrieve stage elapsed time from the original dataset
overheated_50["stage_elapsed_ms"] = test_df.loc[
    overheated_50["row_index"],
    "stage_elapsed_ms"
].values

# Compare detected and missed overheating cases
elapsed_analysis = (
    overheated_50
    .groupby("predicted")["stage_elapsed_ms"]
    .agg(["count", "mean", "std", "min", "max"])
)

elapsed_analysis.index = [
    "Missed Overheated" if i == 0 else "Detected Overheated"
    for i in elapsed_analysis.index
]

display(elapsed_analysis)

,count,mean,std,min,max
Missed Overheated,31,186750.322581,32781.572678,174422,360800
Detected Overheated,133,165994.526316,135419.185987,4800,466000


In [20]:
overheated_50 = results_df[
    (results_df["speed_percent"] == 50) &
    (results_df["condition"] == "overheated")
].copy()

overheated_50["detection_status"] = overheated_50["predicted"].map({
    0: "Missed",
    1: "Detected"
})

# Convert elapsed time from milliseconds to seconds
overheated_50["elapsed_seconds"] = (
    overheated_50["stage_elapsed_ms"] / 1000
)

# Compare temperature across elapsed-time intervals
overheated_50["time_group"] = pd.cut(
    overheated_50["elapsed_seconds"],
    bins=[0, 30, 60, 120, 180, 300, float("inf")],
    labels=["0-30s", "30-60s", "60-120s",
            "120-180s", "180-300s", "300s+"]
)

analysis = overheated_50.groupby(
    ["time_group", "detection_status"],
    observed=False
).agg(
    count=("temperature_c", "count"),
    mean_temperature=("temperature_c", "mean"),
    mean_raw_score=("raw_score", "mean")
).reset_index()

display(analysis)

,time_group,detection_status,count,mean_temperature,mean_raw_score
0,0-30s,Detected,27,30.379630,3.315139
1,0-30s,Missed,0,NaN,NaN
2,30-60s,Detected,4,72.437500,36.979495
3,30-60s,Missed,0,NaN,NaN
4,60-120s,Detected,26,39.221154,9.569913
5,60-120s,Missed,0,NaN,NaN
6,120-180s,Detected,10,27.825000,2.329052
7,120-180s,Missed,18,27.916667,1.432857
8,180-300s,Detected,48,27.687500,2.152149
9,180-300s,Missed,12,27.625000,1.620392


In [21]:
print(test_df.columns.tolist())

['host_received_timestamp_utc', 'device_time_ms', 'stage_elapsed_ms', 'condition', 'motor_running', 'motor_direction', 'speed_percent', 'pwm_duty_8bit', 'ina219_ok', 'power_data_valid', 'bus_voltage_v', 'shunt_voltage_mv', 'current_ma', 'power_mw', 'ds18b20_ok', 'temperature_valid', 'temperature_c', 'mpu6050_ok', 'mpu_data_valid', 'accel_x_m_s2', 'accel_y_m_s2', 'accel_z_m_s2', 'gyro_x_rad_s', 'gyro_y_rad_s', 'gyro_z_rad_s', 'encoder_count', 'encoder_delta', 'encoder_counts_per_second', 'encoder_counts_per_revolution', 'rpm_valid', 'rpm', 'overcurrent_fault', 'last_motor_stop_reason', 'actual']


In [22]:
# Compare condition and actual labels
print("Condition distribution:")
display(test_df["condition"].value_counts(dropna=False))

print("\nActual distribution:")
display(test_df["actual"].value_counts(dropna=False))

print("\nCondition vs Actual:")
display(pd.crosstab(
    test_df["condition"],
    test_df["actual"],
    margins=True
))

Condition distribution:


condition
normal             989
overheated         383
vibration_issue     40
Name: count, dtype: int64


Actual distribution:


actual
0    989
1    423
Name: count, dtype: int64


Condition vs Actual:


actual,0,1,All
condition,,,
normal,989,0,989
overheated,0,383,383
vibration_issue,0,40,40
All,989,423,1412


In [23]:
# Analyze sensor validity for each condition
validity_columns = [
    "ina219_ok",
    "power_data_valid",
    "ds18b20_ok",
    "temperature_valid",
    "mpu6050_ok",
    "mpu_data_valid",
    "rpm_valid"
]

validity_analysis = test_df.groupby("condition")[
    validity_columns
].agg(["count", "sum", "mean"])

display(validity_analysis)

ina219_ok           power_data_valid           ds18b20_ok  \
                    count  sum mean            count  sum mean      count   
condition                                                                   
normal                989  989  1.0              989  989  1.0        989   
overheated            383  383  1.0              383  383  1.0        383   
vibration_issue        40   40  1.0               40   40  1.0         40   

                          temperature_valid  ...      mpu6050_ok            \
                 sum mean             count  ... mean      count  sum mean   
condition                                    ...                             
normal           989  1.0               989  ...  1.0        989  989  1.0   
overheated       383  1.0               383  ...  1.0        383  383  1.0   
vibration_issue   40  1.0                40  ...  1.0         40   40  1.0   

                mpu_data_valid           rpm_valid                 
                         count  sum mean     count  sum      mean  
condition                                                          
normal                     989  989  1.0       989  785  0.793731  
overheated                 383  383  1.0       383   17  0.044386  
vibration_issue             40   40  1.0        40    6  0.150000  

[3 rows x 21 columns]